In [1]:
import numpy as np
import random
from collections import deque
import gymnasium as gym
import tensorflow as tf
from tensorflow.keras import Model, layers
import os

In [2]:

env = gym.make('CartPole-v1')

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

print("State Size: ", state_size)
print("Action Size: ", action_size)

State Size:  4
Action Size:  2


In [3]:
class DQN(Model):
    def __init__(self, action_size, **kwargs):
        super(DQN, self).__init__(**kwargs)
        self.action_size = action_size
        self.d1 = layers.Dense(24, activation='relu', name='d1')
        self.d2 = layers.Dense(24, activation='relu', name='d2')
        self.d3 = layers.Dense(action_size, activation='linear', name='d3')

    def call(self, x):
        x = self.d1(x)
        x = self.d2(x)
        return self.d3(x)

    # Configs for loading the saved model file later on
    def get_config(self):
        config = super(DQN, self).get_config()
        config.update({"action_size": self.action_size})
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [4]:
memory = deque(maxlen=2000)

In [5]:
class Agent:
    def __init__(self, state_size, action_size, gamma=0.99, epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.995, learning_rate=0.001):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.learning_rate = learning_rate

        self.model = self._build_model()
        self.target_model = self._build_model()
        self.update_target_model()

        self.optimizer = tf.keras.optimizers.Adam(learning_rate=self.learning_rate)

    def _build_model(self):
        return DQN(self.action_size)

    def update_target_model(self):
        self.target_model.set_weights(self.model.get_weights())

    def remember(self, state, action, reward, next_state, done):
        memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        q_values = self.model(np.array([state]))
        return np.argmax(q_values[0].numpy())

    def save_model(self, filepath):
        self.model.save(filepath)

    def load_model(self, filepath):
        # Load the saved model from the specified filepath
        self.model = tf.keras.models.load_model(filepath, custom_objects={"DQN": DQN})
        self.target_model = tf.keras.models.load_model(filepath, custom_objects={"DQN": DQN})
        
    def replay(self, batch_size):
        minibatch = random.sample(memory, batch_size)
        for state, action, reward, next_state, done in minibatch:
            with tf.GradientTape() as tape:
                q_values = self.model(np.array([state]), training=True)
                q_value = q_values[0][action]

                if done:
                    target = reward
                else:
                    next_action = np.argmax(self.model(np.array([next_state]))[0].numpy())
                    t = self.target_model(np.array([next_state]))[0][next_action]
                    target = reward + self.gamma * t

                loss = tf.reduce_mean(tf.square(target - q_value))

            grads = tape.gradient(loss, self.model.trainable_variables)
            self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))

In [6]:
batch_size = 32           # Number of samples used for each training step
n_episodes = 500          # Total number of episodes to train on
gamma = 0.95              # Discount factor for future rewards (0 to 1)
epsilon = 1.0             # Initial exploration rate (1 = 100% random actions)
epsilon_min = 0.01        # Minimum exploration rate
epsilon_decay = 0.995     # Decay factor for epsilon after each episode
learning_rate = 0.001     # Step size for neural network weight updates
update_target_every = 5   # Number of episodes between target network updates

In [8]:
import os

base_dir = r"D:\Reinforcement_Learning\src\deep_q_network"
output_dir = os.path.join(base_dir, "cartpole_model")
os.makedirs(output_dir, exist_ok=True)

# Initialize the Agent
agent = Agent(state_size, action_size, gamma=gamma, epsilon=epsilon, epsilon_min=epsilon_min, epsilon_decay=epsilon_decay, learning_rate=learning_rate)
done = False

# Main Script
for e in range(n_episodes):
    state = env.reset()[0]
    state = np.reshape(state, [1, state_size])
    total_reward = 0

    for time_t in range(500):
        action = agent.act(state[0])
        next_state, reward, done, truncated, _ = env.step(action)
        done = done or truncated
        next_state = np.reshape(next_state, [1, state_size])
        agent.remember(state[0], action, reward, next_state[0], done)
        state = next_state
        total_reward += reward

        if done:
            print(f"Episode: {e}/{n_episodes}, Score: {time_t}, Epsilon: {agent.epsilon:.2f}")
            break

    if len(memory) > batch_size:
        loss = agent.replay(batch_size)

    # Update epsilon
    if agent.epsilon > agent.epsilon_min:
        agent.epsilon *= agent.epsilon_decay

    # Update target network
    if e % update_target_every == 0:
        agent.update_target_model()

    
    if e % 100 == 0:
        agent.save_model(os.path.join(output_dir, f"model_{e}.keras"))


agent.save_model(os.path.join(output_dir, f'model_500.keras'))

Episode: 0/500, Score: 17, Epsilon: 1.00
Episode: 1/500, Score: 38, Epsilon: 0.99
Episode: 2/500, Score: 21, Epsilon: 0.99
Episode: 3/500, Score: 24, Epsilon: 0.99
Episode: 4/500, Score: 25, Epsilon: 0.98
Episode: 5/500, Score: 40, Epsilon: 0.98
Episode: 6/500, Score: 22, Epsilon: 0.97
Episode: 7/500, Score: 24, Epsilon: 0.97
Episode: 8/500, Score: 20, Epsilon: 0.96
Episode: 9/500, Score: 23, Epsilon: 0.96
Episode: 10/500, Score: 16, Epsilon: 0.95
Episode: 11/500, Score: 35, Epsilon: 0.95
Episode: 12/500, Score: 11, Epsilon: 0.94
Episode: 13/500, Score: 15, Epsilon: 0.94
Episode: 14/500, Score: 33, Epsilon: 0.93
Episode: 15/500, Score: 13, Epsilon: 0.93
Episode: 16/500, Score: 36, Epsilon: 0.92
Episode: 17/500, Score: 37, Epsilon: 0.92
Episode: 18/500, Score: 29, Epsilon: 0.91
Episode: 19/500, Score: 23, Epsilon: 0.91
Episode: 20/500, Score: 14, Epsilon: 0.90
Episode: 21/500, Score: 17, Epsilon: 0.90
Episode: 22/500, Score: 14, Epsilon: 0.90
Episode: 23/500, Score: 23, Epsilon: 0.89
Ep

In [12]:
BASE_DIR = r"D:\Reinforcement_Learning\src\deep_q_network"
MODEL_DIR = os.path.join(BASE_DIR, "cartpole_model")

def render_episode(agent, model_path, num_episodes=2):
    agent.load_model(model_path)

    env = gym.make("CartPole-v1", render_mode="human")
    try:
        for episode in range(num_episodes):
            state, _ = env.reset()
            state = state.reshape(1, -1)
            done = False
            total_reward = 0

            while not done:
                action = agent.act(state)
                next_state, reward, terminated, truncated, _ = env.step(action)
                state = next_state.reshape(1, -1)
                total_reward += reward
                done = terminated or truncated

            print(f"Episode {episode + 1} reward: {total_reward}")
    finally:
        env.close()

# Initialize agent
state_size = 4
action_size = 2
agent = Agent(state_size, action_size)
agent.epsilon = 0.0  # pure exploitation

# Use an existing saved model
model_path = os.path.join(MODEL_DIR, "model_500.keras")
render_episode(agent, model_path, num_episodes=2)

Episode 1 reward: 131.0
Episode 2 reward: 126.0
